# Supplemental Advanced - Piping Engineering, Support Optimization, and BIM

This notebook combines several advanced topics from the course into one longer workflow: pipe racks, frictional support interfaces, Code_Aster-backed scoring, optional repeated-solve optimization, and IFC exchange.

Use it after Notebooks 06 and 07. The initial scoring and exports use imported Code_Aster result artifacts. Iterative optimization remains disabled unless you intentionally enable repeated Code_Aster solves.


## 1. Model Initialization and Structural Frame Setup

First, we import the core Tuba classes and define the material, piping cross-section (`PipeSection`), and structural H-beam profile (`IBeamSection` loaded from the database).
We build a structural support frame (T-post or portal frame) that our piping system will sit on.


In [ ]:
import sys
from pathlib import Path
import numpy as np

REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tuba import Model
from tuba.model import RectangularSection, IBeamSection, BarSection

# Initialize model with ASME B31.3 standard
model = Model(project_name="Advanced_Industrial_Loop", standard="ASME_B31.3")

# Define piping/structural material
model.add_material(
    "S235JR",
    E=2.1e11,      # Young's Modulus (Pa)
    nu=0.3,        # Poisson's ratio
    rho=7850.0,    # Density (kg/m3)
    allowable_stress={20.0: 137e6, 200.0: 120e6}  # Temp-dependent allowables
)

# Add piping cross-section
model.add_pipe_section("4inch_sch40", OD=0.1143, WT=0.00602)

# Load H-beam section from database
model.add_ibeam_section("HE200B_Girder", "HE200B")

# Define columns using a hollow box section
model.sections["ColBox"] = RectangularSection(
    name="ColBox",
    height_y=0.2,
    height_z=0.2,
    thickness_y=0.008,
    thickness_z=0.008,
)

print("Material and section profiles configured successfully.")

### Building the Support Portal Frame

We build a portal support frame at $x = 5.0\text{m}$ along the pipeline route. It consists of two columns base anchored to the ground and a horizontal HE200B girder beam crossing underneath the pipe.


In [ ]:
# base nodes for columns
c1_base = model.add_node(np.array([5.0, 0.0, -1.0]))
c2_base = model.add_node(np.array([5.0, 0.0,  1.0]))

# frame top joints
c1_top = model.add_node(np.array([5.0, 3.0, -1.0]))
c2_top = model.add_node(np.array([5.0, 3.0,  1.0]))

# Add column elements
model.add_element(id="col_1", type="beam", n1=c1_base, n2=c1_top, section="ColBox", material="S235JR")
model.add_element(id="col_2", type="beam", n1=c2_base, n2=c2_top, section="ColBox", material="S235JR")

# Add horizontal girder beam
model.add_element(id="girder_1", type="beam", n1=c1_top, n2=c2_top, section="HE200B_Girder", material="S235JR")

# Rigidly anchor the base of the columns
model.add_support(node=c1_base, type="anchor")
model.add_support(node=c2_base, type="anchor")

print(f"Portal frame added with anchor supports at nodes {c1_base} and {c2_base}.")


## 2. Modeling the Piping Network and Surrounding Obstacles

Now, we build a 10m piping pipeline that runs parallel to the Z-axis, crossing right over our horizontal girder at $y = 3.1143\text{m}$ (including the shoe offset).
We also register an adjacent cable tray / equipment duct obstacle to verify clash detection and BIM roundtrip.


In [ ]:
# Build piping loop crossing the girder
with model.pipe(section="4inch_sch40", material="S235JR") as p:
    p.start([0.0, 3.1143, 0.0], support="anchor")  # Start anchor
    p.run(5.0)                                      # Run 5m to snapping node crossing the girder
    p.run(5.0)                                      # Run another 5m to end node
    p.end(support="anchor")                         # End anchor

# Retrieve the pipe node directly over the girder
pipe_nodes = [nid for nid, n in model.nodes.items() if np.allclose(n.coords, [5.0, 3.1143, 0.0])]
pipe_crossing_node = pipe_nodes[0]

# Add a physical equipment obstacle nearby
model.add_obstacle(
    id="EquipmentDuct_01",
    type="cuboid",
    min_point=[2.0, 1.0, 0.2],
    max_point=[4.0, 2.5, 1.5]
)

print(f"Piping network built. Crossing point node ID: {pipe_crossing_node}. Obstacle registered.")


## 3. Configuring Realistic Support Interfaces (Non-linear Contact)

Instead of rigid restraints, we configure a realistic sliding shoe rest sitting on the structural beam girder. We model it with unilateral contact (allowing lift-off) and Coulomb friction (steel-on-steel sliding resistance, $\mu = 0.3$).


In [ ]:
# Configure a sliding rest support at the crossing point
model.add_support(
    node=pipe_crossing_node,
    type="rest",
    friction_coefficient=0.3  # Enables non-linear friction and lift-off in STAT_NON_LINE
)

# Define operating thermal and pressure conditions
model.define_load_case(
    name="Operating_Hot",
    gravity=True,
    pressure=2.5e6,     # 2.5 MPa internal pressure
    temperature=220.0,  # 220°C operating temperature
    ref_temperature=20.0
)

print(f"Support rest with friction coefficient 0.3 added to node {pipe_crossing_node}.")


## 4. Code_Aster-Backed Scoring and Optional Optimization

The first evaluation loads or runs Code_Aster once, then scores stress, deflection, support cost, and deformed clash behavior. Repeated optimization loops remain disabled until a configured Code_Aster runtime is available for iterative solves.


In [ ]:
from tuba.analysis.code_aster_notebook import configure_code_aster_notebook_runtime, load_or_run_code_aster_results
from tuba.optimization.objectives import (
    ObjectiveEvaluator,
    StressObjective,
    DeflectionObjective,
    SupportCostObjective,
    ClashObjective,
)
from tuba.optimization.optimizer import RuleBasedSupportPlacer, GeneticSupportPlacer, LLMSupportOptimizer

# Configure objective evaluator
evaluator = ObjectiveEvaluator([
    StressObjective(weight=1.0),
    DeflectionObjective(weight=2.0, max_deflection_m=0.0025), # 2.5mm vertical deflection limit
    SupportCostObjective(weight=0.5),
    ClashObjective(weight=3.0, check_deformed=True),          # Evaluates operating thermal expansion clashes
])

CODE_ASTER_RUNTIME = configure_code_aster_notebook_runtime()
# VS Code/Jupyter review defaults to committed real Code_Aster artifacts; set True only after the runtime doctor passes.
RUN_CODE_ASTER = False
CODE_ASTER_WORK_DIR = REPO_ROOT / "notebooks" / "code_aster_results" / "advanced_operating_hot"

code_aster_run = load_or_run_code_aster_results(
    model,
    "Operating_Hot",
    CODE_ASTER_WORK_DIR,
    run_solver=RUN_CODE_ASTER,
    exec_method=CODE_ASTER_RUNTIME.exec_method,
    wsl_distro=CODE_ASTER_RUNTIME.wsl_distro,
    docker_image=CODE_ASTER_RUNTIME.docker_image,
)
init_results = code_aster_run.results
code_aster_artifact = code_aster_run.artifact

if code_aster_run.ran_solver:
    print("Code_Aster solver executed for this notebook run.")
scores = evaluator.get_detailed_scores(model, init_results)
print("--- Initial Evaluation from Code_Aster Results ---")
print(f"Total Penalty Score: {scores['TotalScore']:.2f}")
print(f"Max deflection: {scores['DeflectionObjective']['details']['max_deflection_mm']:.2f} mm")
print(f"Deflection within limit: {scores['DeflectionObjective']['details']['within_limit']}")

RUN_RULE_BASED_OPTIMIZATION = False
RUN_GENETIC_OPTIMIZATION = False
opt_model = model
opt_results = init_results

if RUN_RULE_BASED_OPTIMIZATION:
    print("\n--- Running Rule-Based Support Placer with Code_Aster ---")
    rule_placer = RuleBasedSupportPlacer(solver_name="code_aster", deflection_limit_m=0.0025)
    opt_model, opt_results = rule_placer.optimize(model, evaluator)
    if opt_results is None:
        raise RuntimeError("Rule-based optimization did not return Code_Aster results.")
    opt_scores = evaluator.get_detailed_scores(opt_model, opt_results)
    print(f"Optimized Score: {opt_scores['TotalScore']:.2f}")
    print(f"Optimized Max Deflection: {opt_scores['DeflectionObjective']['details']['max_deflection_mm']:.2f} mm")
else:
    print("\nRule-based optimization is disabled until the configured Code_Aster runtime is available for iterative solves.")

if RUN_GENETIC_OPTIMIZATION:
    print("\n--- Running Genetic Algorithm Support Placer with Code_Aster ---")
    ga_placer = GeneticSupportPlacer(solver_name="code_aster", population_size=10, generations=15)
    ga_model, ga_results = ga_placer.optimize(model, evaluator)
    if ga_results is None:
        raise RuntimeError("Genetic optimization did not return Code_Aster results.")
    ga_scores = evaluator.get_detailed_scores(ga_model, ga_results)
    print(f"GA Optimized Score: {ga_scores['TotalScore']:.2f}")
    print(f"GA Support Count: {len(ga_model.supports)}")
else:
    print("Genetic optimization is disabled until the configured Code_Aster runtime is available for repeated solves.")

# LLM context uses the imported Code_Aster result state.
llm_opt = LLMSupportOptimizer()
llm_json_context = llm_opt.get_llm_context(model, init_results, evaluator)
print("\n--- LLM JSON Context Snippet ---")
print(llm_json_context[:350] + "...\n")

## 5. Visualizing the Physical supports and Steel Beams

Tuba renders physical support fittings aligned with the piping local coordinates. Standard geometries include:
- **Anchor**: A bolted steel flange.
- **Rest (Shoe)**: T-welded support plates.
- **Guide**: Collar cylinder with bumper plates.
- **Spring Hanger**: Rod and spring canister assembly.

We can export this setup to an interactive 3D HTML page.


In [ ]:
from tuba.plotting.export import export_html

# Export model + FEA stress visual to a standalone HTML file
html_file = "advanced_deformed_stress_view.html"
try:
    export_html(opt_results, html_file, model=opt_model)
    print(f"Interactive 3D HTML exported to: {html_file}")
    print("Open this file in your browser to inspect the 3D H-beams, guide clamps, shoes, and stress heatmaps.")
except Exception as e:
    print(f"Visualization export warning: {e}")


## 6. BIM Data Exchange: Exporting & Importing IFC4 Models

Finally, we serialize our complete model to an **IFC4** model using `IfcExporter` and re-import it using `IfcImporter`. We also inspect the custom FEA stress, support reaction, and friction property sets attached to physical entities.


In [ ]:
from tuba import PlacementAssignment, PlacementFrame

# Preserve IFC-style placement metadata for one representative exported product.
representative = next((e for e in opt_model.elements if e.type in ("pipe_straight", "beam")), None)
if representative is not None:
    frame_id = "advanced_export_frame"
    opt_model.placement_frames[frame_id] = PlacementFrame(
        id=frame_id,
        origin=tuple(float(v) for v in opt_model.nodes[representative.n1].coords),
        frame_type="product",
        source="notebook",
        metadata={"description": "Representative IFC product placement"},
    )
    if not any(a.target == f"element:{representative.id}" and a.role == "object_placement" for a in opt_model.placement_assignments):
        opt_model.placement_assignments.append(
            PlacementAssignment(
                target=f"element:{representative.id}",
                frame=f"placement_frame:{frame_id}",
                role="object_placement",
                source="notebook",
            )
        )
    opt_model.validate()
    print(f"IFC placement frame {frame_id} assigned to {representative.id}.")

from tuba.external.ifc import IfcExporter, IfcImporter
import ifcopenshell

ifc_path = "advanced_piping_rack.ifc"

# 1. Export model to IFC4 with FEA forces and ASME B31.3 stress properties
exporter = IfcExporter()
exporter.export_model(opt_model, ifc_path, results=opt_results)
print(f"IFC model successfully exported to: {ifc_path}")

# 2. Open and inspect properties using raw ifcopenshell queries
ifc_file = ifcopenshell.open(ifc_path)
fasteners = ifc_file.by_type("IfcMechanicalFastener")
print(f"\nExtracted {len(fasteners)} supports from IFC:")
for f in fasteners:
    print(f"  - Support: {f.Name} (Description: {f.Description})")
    # Check custom Tuba property sets
    for definition in f.IsDefinedBy:
        if definition.is_a("IfcRelDefinesByProperties"):
            prop_def = definition.RelatingPropertyDefinition
            if prop_def.is_a("IfcPropertySet") and prop_def.Name == "Pset_TubaSupportForces":
                props = {p.Name: p.NominalValue.wrappedValue for p in prop_def.HasProperties}
                print(f"    Custom Pset_TubaSupportForces properties: {props}")

# 3. Re-import model back into Tuba
importer = IfcImporter()
imported_model = importer.import_model(ifc_path)
print(f"\nRe-imported model summary:")
print(f"  - Pipes: {len([e for e in imported_model.elements if e.type in ('pipe_straight', 'pipe_bend')])}")
print(f"  - Beams: {len([e for e in imported_model.elements if e.type == 'beam'])}")
print(f"  - Supports: {len(imported_model.supports)}")
for s in imported_model.supports:
    print(f"      * Node {s.node}: Type={s.type}, FrictionCoeff={s.friction_coefficient}")
print(f"  - Obstacles: {len(imported_model.obstacles)}")
for obs in imported_model.obstacles:
    print(f"      * Obstacle {obs['id']}: Type={obs['type']}")
